<a href="https://colab.research.google.com/github/GENAURA/Accredian-Enterprize-clone1/blob/main/AI_assessment_Shubham_kumaripynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q chromadb sentence-transformers

In [11]:
import re
import string
import chromadb
from sentence_transformers import SentenceTransformer

print("Libraries loaded successfully")

Libraries loaded successfully


In [15]:
documents = [
    "Python is a programming language used for data science and artificial intelligence.",
    "Machine learning allows systems to learn and improve from experience without being explicitly programmed.",
    "Artificial intelligence is the intelligence of machines or software, as opposed to the intelligence of humans or animals."
]

print(f"Number of documents: {len(documents)}")

Number of documents: 3


In [16]:
def preprocess(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Test the preprocessing function on our documents
cleaned_documents = [preprocess(doc) for doc in documents]
print("Original:", documents[0])
print("Cleaned:", cleaned_documents[0])

Original: Python is a programming language used for data science and artificial intelligence.
Cleaned: python is a programming language used for data science and artificial intelligence


In [17]:
# Load the SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the preprocessed documents
embeddings = model.encode(cleaned_documents)

# Print the embedding for the first document (index 0)
print("Embedding for document 0:")
print(embeddings[0])
print(f"Embedding shape: {embeddings[0].shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding for document 0:
[-5.37488870e-02  2.42363177e-02 -4.75674830e-02  5.85354380e-02
 -2.02419180e-02 -1.51409015e-01  3.05211619e-02  1.89312510e-02
 -5.76515570e-02 -9.21525154e-03 -5.07474504e-02  4.55560684e-02
  1.01700544e-01  1.59673602e-03  5.51950969e-02  1.68126300e-02
 -7.34680966e-02 -1.43093662e-02 -1.24160643e-03 -1.06792636e-01
 -1.43077159e-02  6.68138489e-02 -5.96449375e-02 -4.67674946e-03
  1.34867961e-02 -1.68152191e-02 -8.54191184e-03 -1.05354772e-03
 -3.59437405e-03  2.63005476e-02 -3.36604789e-02  4.77285758e-02
  4.39062566e-02  2.07149703e-02 -1.41080208e-02  1.12657789e-02
 -2.36851890e-02 -5.81075177e-02 -5.90519980e-02  1.68793220e-02
 -5.56988046e-02  2.13495363e-03 -5.45178428e-02 -3.06600202e-02
 -4.33816090e-02  7.93529153e-02 -4.73615043e-02 -1.30617451e-02
  2.10492164e-02  3.06574249e-04 -1.31525993e-01  8.95195734e-03
 -2.52014101e-02 -5.02920784e-02 -1.06437737e-03 -4.12928825e-03
  7.99123645e-02 -2.96058808e-03 -3.93880112e-03 -1.17286488e-01

In [18]:
client = chromadb.Client()
collection = client.get_or_create_collection(name="my_documents")

# Prepare IDs for the documents
doc_ids = [f"id_{i}" for i in range(len(cleaned_documents))]

# Add documents and their embeddings to the collection
collection.add(
    embeddings=embeddings.tolist(),
    documents=cleaned_documents,
    ids=doc_ids
)

print("documents stored successfully")

documents stored successfully


In [19]:
query = input("Enter your question ")

# Preprocess the input query
cleaned_query = preprocess(query)

# Generate query embedding using the same model
query_embedding = model.encode([cleaned_query]).tolist()

# Query the collection for the 1 most relevant document
results = collection.query(
    query_embeddings=query_embedding,
    n_results=1
)

# Extract and print the most relevant document
most_relevant_doc = results["documents"][0][0]
print(f"\nMost relevant document: {most_relevant_doc}")

Enter your question where is used to store and search embeddings

Most relevant document: python is a programming language used for data science and artificial intelligence
